<a href="https://www.kaggle.com/code/eclipse8n/genielablast?scriptVersionId=313091219" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install -q transformers datasets accelerate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import pandas as pd
import numpy as np
import torch
import re

from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

In [3]:
DATA_PATH = "/kaggle/input/competitions/kaz-punct-hackathon/"

train_df = pd.read_csv(DATA_PATH + "train_example.csv")
test_df = pd.read_csv(DATA_PATH + "test.csv")

print("train:", len(train_df), "test:", len(test_df))

train: 500 test: 3552


In [4]:
def strip_and_label(text):

    tokens = text.split()

    labels = []
    clean = []

    for t in tokens:

        if t.endswith("?"):
            labels.append("QUESTION")
            clean.append(t[:-1].lower())

        elif t.endswith((".", "!")):
            labels.append("PERIOD")
            clean.append(t[:-1].lower())

        elif t.endswith(","):
            labels.append("COMMA")
            clean.append(t[:-1].lower())

        else:
            labels.append("O")
            clean.append(t.lower())

    return " ".join(clean), " ".join(labels)

In [5]:
dataset_stream = load_dataset(
    "kz-transformers/multidomain-kazakh-dataset",
    split="train",
    streaming=True
)

README.md: 0.00B [00:00, ?B/s]

In [6]:
rows = []

for i,row in enumerate(dataset_stream):

    text = row["text"]

    sentences = re.split(r"[.!?]", text)

    for s in sentences:

        if s.strip():

            clean, labels = strip_and_label(s.strip())

            if len(clean.split()) == len(labels.split()):

                rows.append({
                    "input_text": clean,
                    "labels": labels
                })

    if len(rows) > 40000:
        break

extra_df = pd.DataFrame(rows)

print("extra sentences:", len(extra_df))

extra sentences: 40001


In [7]:
train_full = pd.concat(
    [train_df[["input_text","labels"]], extra_df],
    ignore_index=True
)

print("total training:", len(train_full))

total training: 40501


In [8]:
LABELS = ["O","COMMA","PERIOD","QUESTION"]

label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

In [9]:
def convert_row(row):

    tokens = row["input_text"].split()
    labels = row["labels"].split()

    return {
        "tokens": tokens,
        "ner_tags": [label2id[l] for l in labels]
    }

dataset = Dataset.from_list([
    convert_row(r) for _,r in train_full.iterrows()
])

dataset = dataset.train_test_split(test_size=0.05)

In [10]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def tokenize(example):

    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    word_ids = tokenized.word_ids()

    labels = []
    prev = None

    for word_id in word_ids:

        if word_id is None:
            labels.append(-100)

        elif word_id != prev:
            labels.append(example["ner_tags"][word_id])

        else:
            labels.append(-100)

        prev = word_id

    tokenized["labels"] = labels

    return tokenized


dataset = dataset.map(tokenize)

Map:   0%|          | 0/38475 [00:00<?, ? examples/s]

Map:   0%|          | 0/2026 [00:00<?, ? examples/s]

In [12]:
model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
training_args = TrainingArguments(

    output_dir="./model",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    logging_steps=200,

    save_strategy="no",

    report_to="none"
)

In [14]:
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dataset["train"],

    eval_dataset=dataset["test"],

    data_collator=data_collator
)

In [15]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
200,0.472175
400,0.252245
600,0.217114
800,0.198389
1000,0.177141
1200,0.179675


TrainOutput(global_step=1203, training_loss=0.24926791578556037, metrics={'train_runtime': 523.9013, 'train_samples_per_second': 73.439, 'train_steps_per_second': 2.296, 'total_flos': 1221216460318824.0, 'train_loss': 0.24926791578556037, 'epoch': 1.0})

In [16]:
def predict(text):

    tokens = text.split()

    encoded = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt"
    )

    encoded = {k:v.to(model.device) for k,v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    preds = outputs.logits.argmax(-1)[0].cpu().numpy()

    word_ids = tokenizer(tokens,is_split_into_words=True).word_ids()

    labels=[]
    prev=None

    for i,w in enumerate(word_ids):

        if w is None:
            continue

        if w!=prev:
            labels.append(id2label[preds[i]])

        prev=w

    return labels

In [17]:
question_words = {"ма","ме","ба","бе","па","пе","қалай","қайда","қашан","неге"}

comma_words = {"бірақ","алайда","сондықтан","өйткені","яғни","демек"}

rows=[]

for _,row in test_df.iterrows():

    tokens=row["input_text"].split()

    preds=predict(row["input_text"])

    if len(preds)<len(tokens):
        preds+=["O"]*(len(tokens)-len(preds))

    if len(preds)>len(tokens):
        preds=preds[:len(tokens)]

    preds[-1]="PERIOD"

    if tokens[-1] in question_words:
        preds[-1]="QUESTION"

    for i in range(len(tokens)-1):
        if tokens[i+1] in comma_words:
            preds[i]="COMMA"

    rows.append({
        "id":row["id"],
        "labels":" ".join(preds)
    })

submission=pd.DataFrame(rows)

submission.to_csv("submission_big_model.csv",index=False)

submission.head()

,id,labels
0,kzp_00001,O O O O COMMA O O O O PERIOD
1,kzp_00002,O O O O O O COMMA O O O O COMMA O O PERIOD
2,kzp_00003,O O O O O O O O O O O O O O O O O O O O O O O ...
3,kzp_00004,O O O O O O O O O O O O O O COMMA O O COMMA O ...
4,kzp_00005,O O O O O O O O O O O O O O O COMMA O PERIOD


In [18]:
print(submission.iloc[2]["labels"])

O O O O O O O O O O O O O O O O O O O O O O O O O O O PERIOD
